# Native reconstruction of Catalyst QSVT

Preserve the source's four phases, degree three, three block bits and six data bits.
The source is fully commented: this runs FlagQuantum, without PennyLane, Catalyst or their optimizer.
Use a local ceiling of eleven physical bits; the source specifies no width budget.
The reflection is negative on zero; both block controls select one. Preserve the big-endian seven-bit Adder word.
The commented source declares nine device wires but uses auxiliary wire nine: this native reconstruction explicitly allocates ten logical wires.
The native [quantum functions](program.py) compose modular updates, a reflection, diagonal phases and an adjoint.
An explicit local CX/U3 pipeline replaces the source CX/RZ/RY/RX plus qjit pipeline; no MLIR, optimizer or timing equivalence is claimed.


In [ ]:
from typing import Annotated

import flagquantum as fq
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.qsvt.catalyst_qsvt.overview.program import (
    apply_qsvt,
)
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.qsvt.catalyst_qsvt.overview.workflow import (
    run_workflow,
)
from flagquantum.synth import Output, QBit, QNum, allocate, qfunc


In [ ]:
source_phases = (
    1.280311896404252,
    8.127145628464149,
    1.8439603212845617,
    -5.002873410775335,
)


@qfunc
def catalyst_reference_qsvt(
    phases: list[float],
    block: Output[Annotated[QNum, 3]],
    data: Output[Annotated[QNum, 6]],
    qsvt_aux: Output[QBit],
) -> None:
    allocate(qsvt_aux)
    allocate(block)
    allocate(data)
    apply_qsvt(phases, data, block, qsvt_aux)


workflow = run_workflow(catalyst_reference_qsvt, phases=source_phases)


Both the native circuit and the converted executable have already been checked against an independent Circuit, including absolute phase and clean compiler work.
The logical block and QSVT auxiliary outputs may remain entangled; they are retained outputs, not compiler work.
The independent reference uses at most a 1 MiB dense matrix for correctness inspection. Native compilation does not use that matrix.


In [ ]:
cost = workflow.resources
print("width", cost.n_qubits, "depth", cost.depth)
print(
    "CX", cost.operation_counts.get("cx", 0), "U3", cost.operation_counts.get("u3", 0)
)
print("synthesis_seconds", workflow.synthesis_seconds)
print("conversion_seconds", workflow.conversion_seconds)
print("local_workflow_seconds", workflow.elapsed_seconds)
result = fq.run(workflow.program)
print(dict(workflow.program.probabilities(result, registers=("qsvt_aux",))))
